In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [2]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [3]:
train = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/train.csv')
test = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv').drop(columns=['id'])

In [4]:
temperature = train['temperature'].astype('float').to_numpy()
voltage = train['voltage'].astype('float').to_numpy()
current = train['current'].astype('float').to_numpy()
irradiance = train['irradiance'].astype('float').to_numpy()
soiling_ratio = train['soiling_ratio'].astype('float').to_numpy()
panel_age = train['panel_age'].astype('float').to_numpy()
maintenance_count = train['maintenance_count'].astype('float').to_numpy()
humidity = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in train['humidity'].to_numpy()])
pressure = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in train['pressure'].to_numpy()])
wind_speed = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in train['wind_speed'].to_numpy()])
pressure = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in train['pressure'].to_numpy()])

In [5]:
#train['ht'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in numpy.log(temperature * humidity)])
train['soiling_impact'] = irradiance * soiling_ratio
#train['wind_resistance'] = numpy.log(wind_speed * pressure)

train['power'] = voltage * current
train['resistance'] = voltage / current
train['vaper_pressure'] = (humidity / 100) * pressure
train['maintenance_frequency'] = maintenance_count / panel_age

In [6]:
temperature = test['temperature'].astype('float').to_numpy()
voltage = test['voltage'].astype('float').to_numpy()
current = test['current'].astype('float').to_numpy()
irradiance = test['irradiance'].astype('float').to_numpy()
soiling_ratio = test['soiling_ratio'].astype('float').to_numpy()
panel_age = test['panel_age'].astype('float').to_numpy()
maintenance_count = test['maintenance_count'].astype('float').to_numpy()
humidity = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in test['humidity'].to_numpy()])
pressure = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in test['pressure'].to_numpy()])
wind_speed = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in test['wind_speed'].to_numpy()])
pressure = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in test['pressure'].to_numpy()])

In [7]:
#test['ht'] = numpy.array([float(i) if len(i.split('.')) >= 2 else 0 for i in numpy.log(temperature * humidity)])
test['soiling_impact'] = irradiance * soiling_ratio
#test['wind_resistance'] = numpy.log(wind_speed * pressure)

test['power'] = voltage * current
test['resistance'] = voltage / current
test['vaper_pressure'] = (humidity / 100) * pressure
test['maintenance_frequency'] = maintenance_count / panel_age

In [8]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [9]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]]=normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        train[i[0]].fillna(train[i[0]].mean(), inplace=True)
        train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]]=normalize.fit_transform(numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))).reshape(-1,1))
    else:
        test[i[0]].fillna(test[i[0]].mean(), inplace=True)
        test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [10]:
train_x = train.drop(columns=['id', 'efficiency'])
train_y = train['efficiency']

In [11]:
LGBM_R_parm = {'colsample_bytree': 0.6657998165699125,
               'drop_rate': 0.8303473371870002,
               'learning_rate': 0.2762739125344964,
               'max_bin': 9983,
               'max_depth': 8623,
               'max_drop': 5480,
               'min_child_samples': 101,
               'min_data_in_leaf': 426,
               'n_estimators': 1628,
               'num_leaves': 3640,
               'objective': 'regression_l1',
               'reg_alpha': 0.39940189926691194,
               'reg_lambda': 0.9567353299218986,
               'skip_drop': 0.6274640597528743,
               'verbosity': -1}

XGB_R_parm = {'colsample_bytree': 0.871893537724492,
              'gamma': 1,
              'learning_rate': 0.923192518624813,
              'max_depth': 15,
              'min_child_weight': 1,
              'n_estimators': 17748,
              'objective': 'binary:logistic',
              'reg_alpha': 45,
              'reg_lambda': 0.8598696247943665,
              'subsample': 0.9109367356405415}

catboost_params = {'iterations' : 1000,
                   'learning_rate': 0.009, 
                   'depth': 5, 
                   'l2_leaf_reg': 5.5,
                   'min_child_samples' : 102,
                   'od_wait' : 50,
                   'random_state' : 42,
                   'eval_metric': 'RMSE', 
                   'bootstrap_type': 'Bayesian', 
                   'grow_policy' : 'Depthwise',
                   'logging_level' : 'Silent'
                 }

X_train, X_test, y_train, y_test = train_test_split(train_x, train_y, test_size=0.1, random_state=100)

In [12]:
LGBM_R = LGBMRegressor(**LGBM_R_parm)

In [13]:
XGB_R = XGBRegressor(**XGB_R_parm)

In [14]:
catboost = CatBoostRegressor(learning_rate=0.009,
                             depth=5,
                             l2_leaf_reg=5.5,
                             min_child_samples=102,
                             grow_policy='Depthwise',
                             iterations=1000,
                             eval_metric='RMSE',
                             od_type='Iter',
                             od_wait=50,
                             random_state=42,
                             logging_level='Silent')

In [15]:
estimators = [
    ('LGBM', make_pipeline(StandardScaler(), LGBM_R)),
    ('xgb', make_pipeline(StandardScaler(), XGB_R)),
    ('cat', make_pipeline(StandardScaler(), catboost))
]
model = StackingRegressor(estimators, final_estimator = RidgeCV())
model.fit(X_train, y_train)

StackingRegressor(estimators=[('LGBM',
                               Pipeline(steps=[('standardscaler',
                                                StandardScaler()),
                                               ('lgbmregressor',
                                                LGBMRegressor(colsample_bytree=0.6657998165699125,
                                                              drop_rate=0.8303473371870002,
                                                              learning_rate=0.2762739125344964,
                                                              max_bin=9983,
                                                              max_depth=8623,
                                                              max_drop=5480,
                                                              min_child_samples=101,
                                                              min_data_in_leaf=426,
                                                              n_estimators=1628,
                                                              num_leaves=3640,
                                                              objective='r...
                                                             max_leaves=None,
                                                             min_child_weight=1,
                                                             missing=nan,
                                                             monotone_constraints=None,
                                                             multi_strategy=None,
                                                             n_estimators=17748,
                                                             n_jobs=None,
                                                             num_parallel_tree=None,
                                                             objective='binary:logistic', ...))])),
                              ('cat',
                               Pipeline(steps=[('standardscaler',
                                                StandardScaler()),
                                               ('catboostregressor',
                                                <catboost.core.CatBoostRegressor object at 0x7a1506f30f90>)]))],
                  final_estimator=RidgeCV())

In [16]:
print(f'Concordance index (lifelines): {100*(1-numpy.sqrt(mean_squared_error(y_test, model.predict(X_test))))}')

Concordance index (lifelines): 90.24493436689819


In [17]:
id = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv')['id']
test_predictions = model.predict(test)
test_predictions

array([0.38940719, 0.52285597, 0.51569913, ..., 0.61056171, 0.43131971,
       0.55122991])

In [18]:
submission = pandas.DataFrame({
    'id': id.values,
    'efficiency': test_predictions,
})
submission

,id,efficiency
0,0,0.389407
1,1,0.522856
2,2,0.515699
3,3,0.478977
4,4,0.467242
...,...,...
11995,11995,0.561570
11996,11996,0.479155
11997,11997,0.610562
11998,11998,0.431320


In [19]:
submission.to_csv('submission.csv', index = False)